# 20260923 Trial Classification

This notebook replaces the older sklearn trial-type notebooks with one pipeline-aware workflow.

The purpose is to label a manageable number of trajectory trials with a GUI, train a classifier from those labels, then predict labels for the rest of the trials so you do not have to hand-label every trajectory.

Expected upstream files:

- `preprocess_out/aligned_session_index.csv`
- `preprocess_out/trial_segment_index.csv`
- `preprocess_out/aligned_sessions/<recording_id>_session.csv`
- `preprocess_out/<recording_id>/behavior/<recording_id>_trials.csv`

## Concepts Covered

This notebook keeps the useful pieces from the older notebooks, but separates them into clean stages:

1. Load the pipeline trial table and aligned session data.
2. Plot one trial trajectory at a time.
3. Label trials with a widget and save labels immediately.
4. Featurize each trial trajectory into fixed-length time-series features plus summary scalars.
5. Train sklearn models from the labeled trials.
6. Compare model performance with held-out labels and a confusion matrix.
7. Predict trial classes for every segmented trial.
8. Review predictions in a second GUI and optionally save corrected labels.

In [ ]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

try:
    import joblib
except ImportError:
    joblib = None

try:
    import cv2
except ImportError:
    cv2 = None

OUTPUT_ROOT = Path("preprocess_out")
CLASSIFICATION_DIR = OUTPUT_ROOT / "trial_classification"
ML_MODEL_DIR = Path("ML_model")
LABELS_CSV = CLASSIFICATION_DIR / "trial_labels.csv"
PREDICTIONS_CSV = CLASSIFICATION_DIR / "trial_type_predictions.csv"
MODEL_PATH = ML_MODEL_DIR / "trial_type_classifier.joblib"

FPS = 30.0
RANDOM_STATE = 0
FIXED_T = 200
INVERT_Y_AXIS = True
SHOW_VIDEO_BACKGROUND = False

LABEL_OPTIONS = [
    "left_small_loop",
    "left_inverse_small_loop",
    "left_big_loop",
    "right_small_loop",
    "right_inverse_small_loop",
    "right_big_loop",
    "non_characteristic",
]

XY_COLUMN_PAIRS = [
    ("ear_mid_x", "ear_mid_y"),
    ("nose.x", "nose.y"),
    ("centroid_x", "centroid_y"),
    ("center_x", "center_y"),
    ("x", "y"),
]

TIME_SERIES_CANDIDATES = [
    "ear_mid_x", "ear_mid_y",
    "nose.x", "nose.y",
    "ear_L.x", "ear_L.y", "ear_R.x", "ear_R.y",
    "head_dir_rad", "ang_vel_speed", "ang_vel_rad_s",
    "in_arena", "in_startbox_L", "in_startbox_R", "arena_only",
    "bpod_any_port_active",
    "bpod_port_1_active", "bpod_port_2_active", "bpod_port_3_active", "bpod_port_4_active",
    "video_any_port_active",
    "video_port_1_active", "video_port_2_active", "video_port_3_active", "video_port_4_active",
]

CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)
ML_MODEL_DIR.mkdir(parents=True, exist_ok=True)

## Load Pipeline Trials

This reads the current pipeline outputs and builds one row per segmented trial. If you are running behavior/SLEAP/Bpod only, that is fine: neural columns are not required here.

In [ ]:
def _read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run the upstream pipeline stage first.")
    return pd.read_csv(path)


def _clean_path(value) -> Path | None:
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return Path(text) if text else None


def load_trial_catalog(output_root: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    aligned_index = _read_csv(output_root / "aligned_session_index.csv")
    trial_index = _read_csv(output_root / "trial_segment_index.csv")

    aligned_lookup = aligned_index.set_index("recording_id")["aligned_csv"].to_dict() if "recording_id" in aligned_index else {}
    trial_rows = []

    for _, row in trial_index.iterrows():
        if str(row.get("status", "")).lower() not in {"segmented", "dry_run"}:
            continue
        trials_csv = _clean_path(row.get("trials_csv"))
        if trials_csv is None or not trials_csv.exists():
            continue

        trials = pd.read_csv(trials_csv)
        recording_id = str(row.get("recording_id"))
        session_id = str(row.get("session_id"))
        aligned_csv = _clean_path(row.get("aligned_csv")) or _clean_path(aligned_lookup.get(recording_id))

        trials["recording_id"] = recording_id
        trials["session_id"] = session_id
        trials["trials_csv"] = str(trials_csv)
        trials["aligned_csv"] = str(aligned_csv) if aligned_csv is not None else None
        trial_rows.append(trials)

    if not trial_rows:
        raise ValueError("No segmented trials found. Run scripts/segment_trials.py first.")

    catalog = pd.concat(trial_rows, ignore_index=True)
    catalog["trial_idx"] = pd.to_numeric(catalog["trial_idx"], errors="coerce").astype("Int64")
    catalog["start_frame"] = pd.to_numeric(catalog["start_frame"], errors="coerce").astype("Int64")
    catalog["end_frame"] = pd.to_numeric(catalog["end_frame"], errors="coerce").astype("Int64")
    catalog["trial_uid"] = catalog["recording_id"].astype(str) + "::" + catalog["trial_idx"].astype(str)
    catalog = catalog.dropna(subset=["trial_idx", "start_frame", "end_frame", "aligned_csv"]).reset_index(drop=True)
    return catalog, aligned_index, trial_index


trial_catalog, aligned_index, trial_index = load_trial_catalog(OUTPUT_ROOT)
print(f"Loaded {len(trial_catalog)} trials from {trial_catalog['recording_id'].nunique()} recordings")
display(trial_catalog.head())

## Shared Plotting and Label Helpers

These helpers are used by both GUIs. Labels are saved after each click so a notebook crash does not wipe out a labeling session.

In [ ]:
aligned_cache: dict[str, pd.DataFrame] = {}
video_frame_cache: dict[str, np.ndarray | None] = {}
undo_stack: list[tuple[str, int]] = []


def load_aligned_for_row(row: pd.Series) -> pd.DataFrame:
    recording_id = str(row["recording_id"])
    if recording_id not in aligned_cache:
        aligned_path = Path(str(row["aligned_csv"]))
        aligned_cache[recording_id] = pd.read_csv(aligned_path)
    return aligned_cache[recording_id]


def select_xy_columns(df: pd.DataFrame) -> tuple[str, str]:
    for x_col, y_col in XY_COLUMN_PAIRS:
        if x_col in df.columns and y_col in df.columns:
            return x_col, y_col
    raise ValueError(f"Could not find x/y columns. Tried: {XY_COLUMN_PAIRS}. Found: {df.columns.tolist()}")


def slice_trial(row: pd.Series) -> pd.DataFrame:
    df = load_aligned_for_row(row)
    start = max(0, int(row["start_frame"]))
    end = min(len(df) - 1, int(row["end_frame"]))
    if end < start:
        return df.iloc[0:0].copy()
    return df.iloc[start:end + 1].copy()


def read_first_video_frame(video_path: str | None):
    if not video_path or pd.isna(video_path) or cv2 is None:
        return None
    video_path = str(video_path)
    if video_path in video_frame_cache:
        return video_frame_cache[video_path]
    cap = cv2.VideoCapture(video_path)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        video_frame_cache[video_path] = None
        return None
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    video_frame_cache[video_path] = frame
    return frame


def plot_trial(row: pd.Series, *, title_prefix: str = "", prediction_text: str | None = None):
    df_trial = slice_trial(row)
    if df_trial.empty:
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.set_title("Empty trial slice")
        return fig

    x_col, y_col = select_xy_columns(df_trial)
    x = pd.to_numeric(df_trial[x_col], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(df_trial[y_col], errors="coerce").to_numpy(dtype=float)
    t = np.arange(len(df_trial))

    fig, ax = plt.subplots(figsize=(7, 6))

    if SHOW_VIDEO_BACKGROUND and "beh_vid_path" in df_trial.columns:
        frame = read_first_video_frame(df_trial["beh_vid_path"].dropna().iloc[0] if df_trial["beh_vid_path"].notna().any() else None)
        if frame is not None:
            ax.imshow(frame)

    ax.plot(x, y, color="0.75", linewidth=1, zorder=1)
    sc = ax.scatter(x, y, c=t, cmap="viridis", s=18, zorder=2)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.any():
        first = np.flatnonzero(valid)[0]
        last = np.flatnonzero(valid)[-1]
        ax.scatter([x[first]], [y[first]], c="lime", edgecolor="black", s=80, label="start", zorder=3)
        ax.scatter([x[last]], [y[last]], c="red", edgecolor="black", s=80, label="end", zorder=3)

    if INVERT_Y_AXIS:
        ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.legend(loc="best")
    plt.colorbar(sc, ax=ax, label="trial frame")

    pieces = [title_prefix.strip()] if title_prefix else []
    pieces.append(f"{row['recording_id']} trial {int(row['trial_idx'])} frames {int(row['start_frame'])}-{int(row['end_frame'])}")
    if pd.notna(row.get("cue_key", np.nan)):
        pieces.append(f"cue={row.get('cue_key')}")
    if prediction_text:
        pieces.append(prediction_text)
    ax.set_title("\n".join(pieces))
    fig.tight_layout()
    return fig


def load_labels() -> pd.DataFrame:
    columns = ["recording_id", "session_id", "trial_idx", "label", "notes", "start_frame", "end_frame"]
    if LABELS_CSV.exists():
        labels = pd.read_csv(LABELS_CSV)
        for col in columns:
            if col not in labels.columns:
                labels[col] = np.nan
        labels["trial_idx"] = pd.to_numeric(labels["trial_idx"], errors="coerce").astype("Int64")
        return labels
    return pd.DataFrame(columns=columns)


def save_labels(labels: pd.DataFrame) -> None:
    CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)
    labels.sort_values(["recording_id", "trial_idx"]).to_csv(LABELS_CSV, index=False)


def upsert_label(labels: pd.DataFrame, row: pd.Series, label: str, notes: str = "") -> pd.DataFrame:
    labels = labels.copy()
    mask = (
        (labels["recording_id"].astype(str) == str(row["recording_id"]))
        & (pd.to_numeric(labels["trial_idx"], errors="coerce") == int(row["trial_idx"]))
    )
    new_row = {
        "recording_id": row["recording_id"],
        "session_id": row.get("session_id"),
        "trial_idx": int(row["trial_idx"]),
        "label": label,
        "notes": notes,
        "start_frame": int(row["start_frame"]),
        "end_frame": int(row["end_frame"]),
    }
    if mask.any():
        for key, value in new_row.items():
            labels.loc[mask, key] = value
    else:
        labels = pd.concat([labels, pd.DataFrame([new_row])], ignore_index=True)
    labels["trial_idx"] = pd.to_numeric(labels["trial_idx"], errors="coerce").astype("Int64")
    save_labels(labels)
    return labels


labels_df = load_labels()
print(f"Current labels: {len(labels_df)} saved in {LABELS_CSV}")

## Labeling GUI

Use this to label enough examples for each trajectory class. `Save label` writes immediately to `trial_labels.csv` and advances to the next visible trial.

In [ ]:
recording_options = sorted(trial_catalog["recording_id"].astype(str).unique())
recording_dropdown = widgets.Dropdown(options=recording_options, description="Recording:", layout=widgets.Layout(width="420px"))
unlabeled_only = widgets.Checkbox(value=True, description="Unlabeled only")
trial_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="Trial:", continuous_update=False)
label_dropdown = widgets.Dropdown(options=LABEL_OPTIONS, value=LABEL_OPTIONS[0], description="Label:", layout=widgets.Layout(width="300px"))
notes_text = widgets.Text(value="", description="Notes:", placeholder="optional", layout=widgets.Layout(width="420px"))
save_button = widgets.Button(description="Save label", button_style="success")
skip_button = widgets.Button(description="Skip", button_style="warning")
back_button = widgets.Button(description="Back")
undo_button = widgets.Button(description="Undo saved", button_style="danger")
reload_button = widgets.Button(description="Reload labels", button_style="info")
status_out = widgets.Output()
plot_out = widgets.Output()

visible_indices: list[int] = []


def labeled_keys() -> set[tuple[str, int]]:
    labels = load_labels()
    return set(zip(labels["recording_id"].astype(str), pd.to_numeric(labels["trial_idx"], errors="coerce").fillna(-1).astype(int)))


def compute_visible_indices() -> list[int]:
    subset = trial_catalog[trial_catalog["recording_id"].astype(str) == str(recording_dropdown.value)]
    indices = subset.index.tolist()
    if unlabeled_only.value:
        keys = labeled_keys()
        indices = [idx for idx in indices if (str(trial_catalog.loc[idx, "recording_id"]), int(trial_catalog.loc[idx, "trial_idx"])) not in keys]
    return indices


def current_row() -> pd.Series | None:
    if not visible_indices:
        return None
    slider_pos = min(int(trial_slider.value), len(visible_indices) - 1)
    return trial_catalog.loc[visible_indices[slider_pos]]


def refresh_options(*_):
    global visible_indices
    visible_indices = compute_visible_indices()
    trial_slider.max = max(0, len(visible_indices) - 1)
    trial_slider.value = min(trial_slider.value, trial_slider.max)
    refresh_plot()


def refresh_plot(*_):
    row = current_row()
    with plot_out:
        clear_output(wait=True)
        if row is None:
            print("No visible trials. Try unchecking 'Unlabeled only' or choose another recording.")
            return
        fig = plot_trial(row, title_prefix="Manual label")
        plt.show(fig)
        plt.close(fig)
    with status_out:
        clear_output(wait=True)
        n_labeled = len(load_labels())
        print(f"Visible {trial_slider.value + 1}/{len(visible_indices)} | saved labels: {n_labeled} | file: {LABELS_CSV}")


def advance():
    if trial_slider.value < trial_slider.max:
        trial_slider.value += 1
    else:
        refresh_options()


def save_current_label(_=None):
    global labels_df
    row = current_row()
    if row is None:
        return
    labels_df = upsert_label(load_labels(), row, label_dropdown.value, notes_text.value)
    undo_stack.append((str(row["recording_id"]), int(row["trial_idx"])))
    notes_text.value = ""
    advance()


def skip_current(_=None):
    advance()


def back_one(_=None):
    if trial_slider.value > 0:
        trial_slider.value -= 1
    else:
        refresh_plot()


def undo_last(_=None):
    if not undo_stack:
        refresh_plot()
        return
    rec, trial_idx = undo_stack.pop()
    labels = load_labels()
    keep = ~((labels["recording_id"].astype(str) == rec) & (pd.to_numeric(labels["trial_idx"], errors="coerce") == trial_idx))
    save_labels(labels.loc[keep].copy())
    refresh_options()


def reload_labels(_=None):
    global labels_df
    labels_df = load_labels()
    refresh_options()


recording_dropdown.observe(refresh_options, names="value")
unlabeled_only.observe(refresh_options, names="value")
trial_slider.observe(refresh_plot, names="value")
save_button.on_click(save_current_label)
skip_button.on_click(skip_current)
back_button.on_click(back_one)
undo_button.on_click(undo_last)
reload_button.on_click(reload_labels)

controls = widgets.VBox([
    widgets.HBox([recording_dropdown, unlabeled_only]),
    widgets.HBox([trial_slider]),
    widgets.HBox([label_dropdown, notes_text]),
    widgets.HBox([save_button, skip_button, back_button, undo_button, reload_button]),
    status_out,
    plot_out,
])

display(controls)
refresh_options()

## Feature Extraction

The model cannot learn from raw variable-length trajectories directly. This section converts each trial into a fixed-length vector:

- resampled trajectory/body time series
- duration and geometry scalars
- ROI occupancy fractions
- Bpod port-active fractions when present

In [ ]:
@dataclass
class TrialFeatureResult:
    X: np.ndarray
    meta: pd.DataFrame
    feature_names: list[str]


class TrialFeaturizer:
    def __init__(self, fixed_t: int = FIXED_T, fps: float = FPS):
        self.fixed_t = int(fixed_t)
        self.fps = float(fps)
        self.time_series_cols_: list[str] | None = None
        self.scalar_names_: list[str] | None = None

    def prepare_schema(self, catalog: pd.DataFrame) -> None:
        available = set()
        for _, row in catalog.drop_duplicates("recording_id").iterrows():
            df = load_aligned_for_row(row)
            available.update(df.columns)

        cols = [col for col in TIME_SERIES_CANDIDATES if col in available]
        active_port_cols = sorted(
            col for col in available
            if col.endswith("_active")
            and (col.startswith("bpod_port_") or col.startswith("video_port_"))
        )
        extra_ports = [col for col in active_port_cols if col not in cols]
        self.time_series_cols_ = [*cols, *extra_ports]

        scalar_names = [
            "duration_s", "duration_frames",
            "path_length_px", "displacement_px", "straightness",
            "x_range_px", "y_range_px", "start_x", "start_y", "end_x", "end_y", "mean_speed_px_s",
            "frac_in_arena", "frac_in_startbox_L", "frac_in_startbox_R", "frac_arena_only", "frac_bpod_any_port_active", "frac_video_any_port_active",
            "start_side_L", "start_side_R", "end_side_L", "end_side_R",
        ]
        scalar_names.extend(f"frac_{col}" for col in active_port_cols)
        self.scalar_names_ = list(dict.fromkeys(scalar_names))

    def numeric_series(self, values) -> np.ndarray:
        arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
        if len(arr) == 0:
            return np.full(self.fixed_t, np.nan)
        valid = np.isfinite(arr)
        if valid.sum() == 0:
            return np.full(self.fixed_t, np.nan)
        x_old = np.linspace(0, 1, len(arr))
        x_new = np.linspace(0, 1, self.fixed_t)
        return np.interp(x_new, x_old[valid], arr[valid])

    def scalar_features(self, df_trial: pd.DataFrame, row: pd.Series) -> dict[str, float]:
        scalars: dict[str, float] = {}
        scalars["duration_s"] = float(row.get("duration_s", np.nan)) if pd.notna(row.get("duration_s", np.nan)) else len(df_trial) / self.fps
        scalars["duration_frames"] = float(len(df_trial))

        try:
            x_col, y_col = select_xy_columns(df_trial)
            x = pd.to_numeric(df_trial[x_col], errors="coerce").to_numpy(dtype=float)
            y = pd.to_numeric(df_trial[y_col], errors="coerce").to_numpy(dtype=float)
            valid = np.isfinite(x) & np.isfinite(y)
            if valid.sum() >= 2:
                xv = x[valid]
                yv = y[valid]
                steps = np.sqrt(np.diff(xv) ** 2 + np.diff(yv) ** 2)
                path_length = float(np.nansum(steps))
                displacement = float(np.sqrt((xv[-1] - xv[0]) ** 2 + (yv[-1] - yv[0]) ** 2))
                scalars.update({
                    "path_length_px": path_length,
                    "displacement_px": displacement,
                    "straightness": displacement / path_length if path_length > 0 else np.nan,
                    "x_range_px": float(np.nanmax(xv) - np.nanmin(xv)),
                    "y_range_px": float(np.nanmax(yv) - np.nanmin(yv)),
                    "start_x": float(xv[0]),
                    "start_y": float(yv[0]),
                    "end_x": float(xv[-1]),
                    "end_y": float(yv[-1]),
                    "mean_speed_px_s": float(path_length / scalars["duration_s"]) if scalars["duration_s"] > 0 else np.nan,
                })
        except Exception:
            pass

        for col in ["in_arena", "in_startbox_L", "in_startbox_R", "arena_only", "bpod_any_port_active", "video_any_port_active"]:
            if col in df_trial.columns:
                scalars[f"frac_{col}"] = float(pd.to_numeric(df_trial[col], errors="coerce").mean())

        for col in [c for c in df_trial.columns if c.endswith("_active") and (c.startswith("bpod_port_") or c.startswith("video_port_"))]:
            scalars[f"frac_{col}"] = float(pd.to_numeric(df_trial[col], errors="coerce").mean())

        for side_col in ["start_side", "end_side"]:
            value = row.get(side_col)
            scalars[f"{side_col}_L"] = float(value == "L")
            scalars[f"{side_col}_R"] = float(value == "R")

        return scalars

    def featurize_one(self, row: pd.Series) -> tuple[np.ndarray, list[str]]:
        if self.time_series_cols_ is None or self.scalar_names_ is None:
            raise RuntimeError("Call prepare_schema before featurize_one")

        df_trial = slice_trial(row)
        series_features = []
        series_names = []
        for col in self.time_series_cols_:
            values = df_trial[col] if col in df_trial.columns else pd.Series([np.nan] * len(df_trial))
            series_features.append(self.numeric_series(values))
            series_names.extend([f"{col}_t{i:03d}" for i in range(self.fixed_t)])

        scalars = self.scalar_features(df_trial, row)
        scalar_values = np.array([scalars.get(name, np.nan) for name in self.scalar_names_], dtype=float)

        if series_features:
            vector = np.concatenate([np.concatenate(series_features), scalar_values])
        else:
            vector = scalar_values
        return vector, [*series_names, *self.scalar_names_]

    def build_matrix(self, catalog: pd.DataFrame) -> TrialFeatureResult:
        self.prepare_schema(catalog)
        vectors = []
        metas = []
        feature_names = None
        skipped = []

        for _, row in catalog.iterrows():
            try:
                vector, names = self.featurize_one(row)
                if feature_names is None:
                    feature_names = names
                vectors.append(vector)
                metas.append({
                    "recording_id": row["recording_id"],
                    "session_id": row.get("session_id"),
                    "trial_idx": int(row["trial_idx"]),
                    "start_frame": int(row["start_frame"]),
                    "end_frame": int(row["end_frame"]),
                    "trial_uid": row["trial_uid"],
                })
            except Exception as exc:
                skipped.append({"trial_uid": row.get("trial_uid"), "error": repr(exc)})

        if skipped:
            skipped_df = pd.DataFrame(skipped)
            skipped_path = CLASSIFICATION_DIR / "feature_skipped_trials.csv"
            skipped_df.to_csv(skipped_path, index=False)
            print(f"Skipped {len(skipped_df)} trials; wrote {skipped_path}")

        if not vectors:
            raise ValueError("No trial features could be built")
        return TrialFeatureResult(np.vstack(vectors), pd.DataFrame(metas), feature_names or [])


featurizer = TrialFeaturizer(fixed_t=FIXED_T, fps=FPS)
features = featurizer.build_matrix(trial_catalog)
print(features.X.shape)
print(f"Feature count: {len(features.feature_names)}")
print("Time-series columns:", featurizer.time_series_cols_)
display(features.meta.head())

## Train and Compare Models

Run this after you have saved labels. It trains several sklearn options from the old notebooks: linear SVM, logistic regression, and random forest.

In [ ]:
def labels_for_features(meta: pd.DataFrame, labels: pd.DataFrame) -> pd.DataFrame:
    labels = labels.copy()
    labels["trial_idx"] = pd.to_numeric(labels["trial_idx"], errors="coerce").astype("Int64")
    merged = meta.merge(
        labels[["recording_id", "trial_idx", "label"]],
        on=["recording_id", "trial_idx"],
        how="left",
    )
    return merged


label_map = labels_for_features(features.meta, load_labels())
labeled_mask = label_map["label"].notna()
X_labeled = features.X[labeled_mask.to_numpy()]
y_labeled = label_map.loc[labeled_mask, "label"].astype(str).to_numpy()
meta_labeled = label_map.loc[labeled_mask].reset_index(drop=True)

print(f"Labeled trials available for training: {len(y_labeled)}")
display(pd.Series(y_labeled).value_counts().rename("n").to_frame())

models = {
    "linear_svc": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearSVC(class_weight="balanced", C=1.0, max_iter=20000, random_state=RANDOM_STATE)),
    ]),
    "logreg": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)),
    ]),
}

model_scores = []
fitted_models = {}
heldout = None

if len(y_labeled) < 4 or pd.Series(y_labeled).nunique() < 2:
    print("Need at least two classes and a few labeled trials before model comparison is meaningful.")
else:
    counts = pd.Series(y_labeled).value_counts()
    stratify = y_labeled if counts.min() >= 2 else None
    test_size = 0.25 if len(y_labeled) >= 12 else 0.33
    Xt, Xv, yt, yv, mt, mv = train_test_split(
        X_labeled,
        y_labeled,
        meta_labeled,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=stratify,
    )
    heldout = (Xv, yv, mv)

    for name, model in models.items():
        fitted = clone(model).fit(Xt, yt)
        pred = fitted.predict(Xv)
        score = balanced_accuracy_score(yv, pred)
        fitted_models[name] = fitted
        model_scores.append({"model": name, "balanced_accuracy": score})
        print("\n===", name, "===")
        print(classification_report(yv, pred, zero_division=0))

    scores_df = pd.DataFrame(model_scores).sort_values("balanced_accuracy", ascending=False)
    display(scores_df)

    best_name = scores_df.iloc[0]["model"]
    print(f"Best held-out model: {best_name}")

## Confusion Matrix

This shows held-out errors for the best model from the comparison section.

In [ ]:
if "scores_df" not in globals() or scores_df.empty or heldout is None:
    print("No held-out model comparison yet.")
else:
    Xv, yv, mv = heldout
    best_name = scores_df.iloc[0]["model"]
    pred = fitted_models[best_name].predict(Xv)
    labels_order = sorted(pd.unique(np.concatenate([yv, pred])))
    cm = confusion_matrix(yv, pred, labels=labels_order)
    disp = ConfusionMatrixDisplay(cm, display_labels=labels_order)
    fig, ax = plt.subplots(figsize=(8, 7))
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    ax.set_title(f"Held-out confusion matrix: {best_name}")
    fig.tight_layout()
    plt.show()

## Fit Final Model and Predict All Trials

Choose the model name here. `auto` uses the best held-out model if one exists; otherwise it defaults to logistic regression.

In [ ]:
FINAL_MODEL_NAME = "auto"  # "auto", "linear_svc", "logreg", or "random_forest"


def predict_confidence(model, X: np.ndarray) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        return np.nanmax(proba, axis=1)
    if hasattr(model, "decision_function"):
        decision = model.decision_function(X)
        decision = np.asarray(decision)
        if decision.ndim == 1:
            return 1 / (1 + np.exp(-np.abs(decision)))
        best = np.nanmax(decision, axis=1)
        second = np.partition(decision, -2, axis=1)[:, -2] if decision.shape[1] > 1 else 0
        return 1 / (1 + np.exp(-(best - second)))
    return np.full(X.shape[0], np.nan)


if len(y_labeled) < 2 or pd.Series(y_labeled).nunique() < 2:
    raise ValueError("Not enough labeled data to fit a final classifier yet.")

if FINAL_MODEL_NAME == "auto":
    if "scores_df" in globals() and not scores_df.empty:
        final_name = str(scores_df.iloc[0]["model"])
    else:
        final_name = "logreg"
else:
    final_name = FINAL_MODEL_NAME

final_model = clone(models[final_name]).fit(X_labeled, y_labeled)
pred_label = final_model.predict(features.X)
pred_conf = predict_confidence(final_model, features.X)

predictions = features.meta.copy()
predictions["pred_label"] = pred_label
predictions["pred_conf"] = pred_conf
predictions = predictions.merge(
    load_labels()[["recording_id", "trial_idx", "label"]].rename(columns={"label": "manual_label"}),
    on=["recording_id", "trial_idx"],
    how="left",
)

CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)
ML_MODEL_DIR.mkdir(parents=True, exist_ok=True)
predictions.to_csv(PREDICTIONS_CSV, index=False)
print(f"Wrote {PREDICTIONS_CSV}")

if joblib is not None:
    joblib.dump(
        {
            "model_name": final_name,
            "model": final_model,
            "feature_names": features.feature_names,
            "fixed_t": FIXED_T,
            "fps": FPS,
            "time_series_cols": featurizer.time_series_cols_,
            "scalar_names": featurizer.scalar_names_,
            "feature_version": "trial_features_v2_bpod_video_ports",
            "label_options": LABEL_OPTIONS,
        },
        MODEL_PATH,
    )
    print(f"Wrote {MODEL_PATH}")
else:
    print("joblib is not installed; skipped model export")

display(predictions.head())

## Prediction Review GUI

Use this after prediction. Filter by predicted label, inspect trials, and save corrections directly back into `trial_labels.csv`. This is the part that lets the classifier reduce manual labeling instead of replacing your judgment blindly.

In [ ]:
if not PREDICTIONS_CSV.exists():
    raise FileNotFoundError(f"Run the prediction section first: {PREDICTIONS_CSV}")

pred_df = pd.read_csv(PREDICTIONS_CSV)
pred_recordings = sorted(pred_df["recording_id"].astype(str).unique())
pred_labels = ["(all)"] + sorted(pred_df["pred_label"].dropna().astype(str).unique())

pred_recording_dropdown = widgets.Dropdown(options=pred_recordings, description="Recording:", layout=widgets.Layout(width="420px"))
pred_filter_dropdown = widgets.Dropdown(options=pred_labels, value="(all)", description="Pred:")
pred_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="Trial:", continuous_update=False)
correction_dropdown = widgets.Dropdown(options=LABEL_OPTIONS, value=LABEL_OPTIONS[0], description="Correct:", layout=widgets.Layout(width="300px"))
correction_notes = widgets.Text(value="", description="Notes:", placeholder="optional", layout=widgets.Layout(width="420px"))
accept_button = widgets.Button(description="Accept pred", button_style="success")
correct_button = widgets.Button(description="Save correction", button_style="warning")
pred_status_out = widgets.Output()
pred_plot_out = widgets.Output()

pred_visible_indices: list[int] = []


def pred_visible() -> list[int]:
    subset = pred_df[pred_df["recording_id"].astype(str) == str(pred_recording_dropdown.value)]
    if pred_filter_dropdown.value != "(all)":
        subset = subset[subset["pred_label"].astype(str) == str(pred_filter_dropdown.value)]
    return subset.index.tolist()


def pred_current_row() -> pd.Series | None:
    if not pred_visible_indices:
        return None
    return pred_df.loc[pred_visible_indices[min(pred_slider.value, len(pred_visible_indices) - 1)]]


def pred_catalog_row(pred_row: pd.Series) -> pd.Series:
    match = trial_catalog[
        (trial_catalog["recording_id"].astype(str) == str(pred_row["recording_id"]))
        & (pd.to_numeric(trial_catalog["trial_idx"], errors="coerce") == int(pred_row["trial_idx"]))
    ]
    if match.empty:
        raise KeyError("Prediction row is not present in trial_catalog")
    return match.iloc[0]


def refresh_pred_options(*_):
    global pred_visible_indices
    pred_visible_indices = pred_visible()
    pred_slider.max = max(0, len(pred_visible_indices) - 1)
    pred_slider.value = min(pred_slider.value, pred_slider.max)
    refresh_pred_plot()


def refresh_pred_plot(*_):
    row = pred_current_row()
    with pred_plot_out:
        clear_output(wait=True)
        if row is None:
            print("No predictions for this filter.")
            return
        catalog_row = pred_catalog_row(row)
        pred_text = f"pred={row.get('pred_label')} conf={row.get('pred_conf', np.nan):.2f} manual={row.get('manual_label', '')}"
        fig = plot_trial(catalog_row, title_prefix="Prediction review", prediction_text=pred_text)
        plt.show(fig)
        plt.close(fig)
    with pred_status_out:
        clear_output(wait=True)
        print(f"Visible {pred_slider.value + 1}/{len(pred_visible_indices)} | predictions: {PREDICTIONS_CSV}")


def pred_advance():
    if pred_slider.value < pred_slider.max:
        pred_slider.value += 1
    else:
        refresh_pred_options()


def accept_prediction(_=None):
    row = pred_current_row()
    if row is None:
        return
    catalog_row = pred_catalog_row(row)
    upsert_label(load_labels(), catalog_row, str(row["pred_label"]), correction_notes.value or "accepted prediction")
    pred_advance()


def save_correction(_=None):
    row = pred_current_row()
    if row is None:
        return
    catalog_row = pred_catalog_row(row)
    upsert_label(load_labels(), catalog_row, correction_dropdown.value, correction_notes.value)
    pred_advance()


pred_recording_dropdown.observe(refresh_pred_options, names="value")
pred_filter_dropdown.observe(refresh_pred_options, names="value")
pred_slider.observe(refresh_pred_plot, names="value")
accept_button.on_click(accept_prediction)
correct_button.on_click(save_correction)

display(widgets.VBox([
    widgets.HBox([pred_recording_dropdown, pred_filter_dropdown]),
    pred_slider,
    widgets.HBox([correction_dropdown, correction_notes]),
    widgets.HBox([accept_button, correct_button]),
    pred_status_out,
    pred_plot_out,
]))
refresh_pred_options()

## Output Files

This notebook writes classification-specific files under:

```text
preprocess_out/trial_classification/
```

Important files:

- `trial_labels.csv`: manual labels and accepted/corrected prediction labels.
- `trial_type_predictions.csv`: model predictions for every segmented trial.
- `ML_model/trial_type_classifier.joblib`: exported sklearn model bundle, when `joblib` is available.
- `feature_skipped_trials.csv`: trials skipped during feature extraction, if any.
- Run `scripts/predict_trial_labels.py` to apply the saved model to later processed recordings.
- Open `20260923_trial_prediction_review.ipynb` for student review/correction of saved-model predictions.